In [ ]:
using DelimitedFiles
using Distances
using FileIO
using DataFrames
using LinearAlgebra
using Random
using StatsBase
using Statistics
using CSV
using PlotlyJS

In [ ]:
using PyCall
m21 = pyimport("music21")
np = pyimport("numpy")

In [ ]:
const msr2 = import MusicSpiralRepresentation

### Utility functions

In [ ]:
function get_key_lims(piece_ann)
    local lk
    try
        lk = piece_ann[!, :local_key]
    catch
        lk = piece_ann[!, :localkey]
    end
    lims = Int64[]
    for i in 2:length(lk)
        if lk[i-1] != lk[i]
            push!(lims, i)
        end
    end

    # Get the quarterbeats values
    quarterbeats_strings = piece_ann[lims, :quarterbeats]

    # Convert strings to real numbers
    quarterbeats_real = Float64[]
    for qb_str in quarterbeats_strings
        if contains(string(qb_str), "/")
            # Handle fraction strings like "11/2"
            parts = split(string(qb_str), "/")
            numerator = parse(Float64, parts[1])
            denominator = parse(Float64, parts[2])
            push!(quarterbeats_real, numerator / denominator)
        else
            # Handle integer strings like "10"
            push!(quarterbeats_real, parse(Float64, string(qb_str)))
        end
    end

    return quarterbeats_real
end
function process_tsv_score(df; repeats=false)

    # Calculate end times for each note
    # end = start + duration
    if repeats
        measure_number = df.mc
        beats = df.quarterbeats_all_endings
    else
        measure_number = df.mn
        beats = df.quarterbeats
    end
    qbeats = frac_to_float(beats)
    df.end_time = qbeats .+ convert.(Float32, df.duration_qb)

    # Create the new DataFrame with selected columns
    result = DataFrame(
        Measure=measure_number,
        TimeSignature=df.timesig,
        StartQuarter=qbeats,
        EndQuarter=df.end_time,
        Duration=df.duration_qb,
        Pitch=df.midi
    )

    # Sort by measure and start time
    sort!(result, [:Measure, :StartQuarter])

    return result
end

function get_key_accuracy_by_window(df_piece, piece_ann; w_s=20, return_keys=false, mean_center=true)
        mc_ann = groupby(piece_ann, :mc)
        ann_k_measures = Dict{Int,Any}()
        for i in 1:length(mc_ann)
            ann_k_measures[mc_ann[i][1, :mc]] = mc_ann[i][!, :localkey][1]
        end
        # divide the piece into overlapping windows of size w_s with an overlap of w_s-1
        w_piece = get_oswindows(df_piece, w_s, w_s - 1)
        p_keys = Vector{String}(undef, length(w_piece))

        #computing the center of effect for each window and getting the closest key
        for i in 1:length(w_piece)
            p_keys[i] = msr2.get_distance_to_keys(msr2.get_center_effect(Matrix(w_piece[i])))[1, 1]
        end
        # defining the main key as the most frequent and getting the relative keys
        main_key = msr2.get_rank_freq(p_keys)[1, 1]
        rel_keys = msr2.funhar_seq(p_keys, main_key)
        ce_ix = zeros(Int64, length(rel_keys))
        ann_keys = Array{Any}(undef, length(rel_keys))
        for i in 1:length(rel_keys)
            if mean_center
                ce_ix[i] = Int(round(mean(w_piece[i][!, :Measure])))
            else
                # Get center row of window
                center_row_idx = div(nrow(w_piece[i]), 2) + 1
                # Get quarterbeat at center (using start of center note)
                center_qb = w_piece[i][center_row_idx, :StartQuarter]
                # Find which measure contains this quarterbeat
                measure_idx = findfirst(row -> row.StartQuarter <= center_qb <= row.EndQuarter, eachrow(df_piece))
                ce_ix[i] = df_piece[measure_idx, :Measure]
            end
            try
                ann_keys[i] = Array(piece_ann[findall(x -> x == ce_ix[i], piece_ann[!, :mc]), :localkey])
            catch
                ann_keys[i] = String(piece_ann[findall(x -> x == ce_ix[i], piece_ann[!, :mc]), :localkey])
            end
        end

        u_measures = unique(ce_ix)
        guess_k_measures = Dict{Int,Any}()
        for uv in u_measures
            ix = findall(x -> x == uv, ce_ix)
            guess_k_measures[uv] = msr2.get_rank_freq(rel_keys[ix])[1, 1]
        end
        # calculating the accuracy of the key prediction
        tot_measures = 0
        correct_measures = 0
        for key in keys(ann_k_measures)
            if haskey(guess_k_measures, key)
                tot_measures += 1
                if ann_k_measures[key] == guess_k_measures[key]
                    correct_measures += 1
                end
            end
        end
        if return_keys
            return tot_measures, correct_measures, correct_measures / tot_measures, rel_keys
        else
            return tot_measures, correct_measures, correct_measures / tot_measures
        end
end

function get_key_accuracy_by_measure(df_piece, df_ann; w_s=20)
        piece_measures = groupby(df_piece, :Measure)
        grouped_ann = groupby(df_ann, :mn)
        pw = get_oswindows(df_piece, w_s, w_s - 1)
        p_keys = Vector{String}(undef, length(pw))
        for i in 1:length(pw)
            p_keys[i] = msr2.get_distance_to_keys(msr2.get_center_effect(Matrix(pw[i])))[1, 1]
        end
        piece_key = msr2.get_rank_freq(p_keys)[1, 1]
        key_ann = Dict{Int,Vector{String}}()
        for i in 1:length(grouped_ann)
            keys = vcat(convert(Vector{String}, collect(skipmissing(grouped_ann[i][!, :numeral]))), convert(Vector{String}, grouped_ann[i][!, :localkey]))
            num_me = grouped_ann[i][1, :mn]
            key_ann[num_me] = keys
        end
        key_guess = Dict{Int,Vector{String}}()
        for i in 1:length(piece_measures)
            nm = piece_measures[i][1, :Measure]
            k_m = msr2.get_distance_to_keys(msr2.get_center_effect(Matrix(piece_measures[i])))[1:3, 1]
            key_guess[nm] = msr2.funhar_seq(k_m, piece_key)
        end
        gg = 0 #1st key correct guesses
        gg2 = 0 #1st and 2nd key correct guesses
        gg3 = 0 #1st, 2nd and 3rd key correct guesses
        err_counter = 0
        key_0 = 0
        for key in keys(key_ann)
            if key == 0
                key_0 = 1
                continue
            end
            try
                if !isnothing(findfirst(x -> x == key_guess[key][1], key_ann[key]))
                    gg += 1
                    gg2 += 1
                    gg3 += 1
                elseif !isnothing(findfirst(x -> x == key_guess[key][2], key_ann[key]))
                    gg2 += 1
                    gg3 += 1
                elseif !isnothing(findfirst(x -> x == key_guess[key][3], key_ann[key]))
                    gg3 += 1
                end
            catch e
                err_counter += 1
                println("Measure $key: Not found in key_guess.")
            end
        end
        return gg / (length(keys(key_ann)) - key_0), gg2 / (length(keys(key_ann)) - key_0), gg3 / (length(keys(key_ann)) - key_0), err_counter
    end

    function get_chord_accuracy_by_measure(df_piece, df_ann; w_s=20, include_keys=false)
        piece_measures = groupby(df_piece, :Measure)
        grouped_ann = groupby(df_ann, :mn)

        # Determine global key from overlapping windows
        pw = get_oswindows(df_piece, w_s, w_s - 1)
        p_keys = Vector{String}(undef, length(pw))
        for i in 1:length(pw)
            p_keys[i] = msr2.get_distance_to_keys(msr2.get_center_effect(Matrix(pw[i])))[1, 1]
        end
        global_key = msr2.get_rank_freq(p_keys)[1, 1]

        # Build annotation dictionary
        chord_ann = Dict{Int,Vector{String}}()
        for i in 1:length(grouped_ann)
            if include_keys
                chords = vcat(convert(Vector{String}, collect(skipmissing(grouped_ann[i][!, :numeral]))),
                    convert(Vector{String}, grouped_ann[i][!, :localkey]))
            else
                chords = convert(Vector{String}, collect(skipmissing(grouped_ann[i][!, :numeral])))
            end
            num_me = grouped_ann[i][1, :mn]
            chord_ann[num_me] = chords
        end

        # Build prediction dictionary with top-3 chords per measure
        chord_guess = Dict{Int,Vector{String}}()
        for i in 1:length(piece_measures)
            nm = piece_measures[i][1, :Measure]
            c_m = msr2.get_distance_to_chords(msr2.get_center_effect(Matrix(piece_measures[i])))[1:3, 1]
            chord_guess[nm] = msr2.funhar_seq(c_m, global_key)
        end

        # Calculate accuracy
        gg = 0  # 1st chord correct
        gg2 = 0 # 1st or 2nd chord correct
        gg3 = 0 # 1st, 2nd, or 3rd chord correct
        err_counter = 0
        key_0 = 0

        for key in keys(chord_ann)
            if key == 0
                key_0 = 1
                continue
            end
            try
                if !isnothing(findfirst(x -> x == chord_guess[key][1], chord_ann[key]))
                    gg += 1
                    gg2 += 1
                    gg3 += 1
                elseif !isnothing(findfirst(x -> x == chord_guess[key][2], chord_ann[key]))
                    gg2 += 1
                    gg3 += 1
                elseif !isnothing(findfirst(x -> x == chord_guess[key][3], chord_ann[key]))
                    gg3 += 1
                end
            catch e
                err_counter += 1
                println("Measure $key: Not found in chord_guess.")
            end
        end

        return gg / (length(keys(chord_ann)) - key_0), gg2 / (length(keys(chord_ann)) - key_0),
        gg3 / (length(keys(chord_ann)) - key_0), err_counter
end

# Ground-truth annotations

In [ ]:
moz_ann_path = "/DCML_Corpora/mozart_piano_sonatas-main/harmonies" #annotated data
moz_ann_list = readdir(moz_ann_path)
moz_ann_list = filter(x -> occursin(r"\.tsv$", x), moz_ann_list)
moz_ann_names = map(x -> split(x, ".")[1], moz_ann_list)
moz_piece_ann = map(x -> CSV.read(joinpath(moz_ann_path, moz_ann_list[x]), DataFrame, delim='\t', header=true), 1:length(moz_ann_list));

# Scores

In [ ]:
mozart_folder = "/DCML_Corpora/mozart_piano_sonatas-main/notes/"
mozart_list = readdir(mozart_folder)
mozart_list = filter(x -> endswith(x, ".tsv"), mozart_list);

In [ ]:
#saving names 
#bee_names = map(x -> split(x, ".")[1], bee_list);
mozart_names = map(x -> split(x, ".")[1], mozart_list);

In [ ]:
moz_dfs = DataFrame[]
good_ix = Int64[]
for i in 1:length(mozart_list)
    try
        tsv_piece = CSV.read(joinpath(mozart_folder, mozart_list[i]), DataFrame, delim='\t', header=true)
        df_piece = process_tsv_score(tsv_piece, repeats=true)
        push!(moz_dfs, df_piece)
        push!(good_ix, i)
    catch e
        println("Error processing $i: $e")
    end
end

# Two types of accuracy:

## 1) Accuracy by measure - very local

* Is the accuracy defined by the center of effect inside the measure, e.g. using the notes that are exclusively inside the measure bars
* The resulting key (in numeral, relative to the fundamental), is compared to the "local key" and their "relative chords" that serve as local relative keys
* For the guessed key in measure "i" (gk_i) we compare them with the "local key" and "relative chords" annotated inside the measure
* Second and third guesses, if the first ranked key is not in the set of keys, the second most likely key is then compared, following by the third.

## 2) Accuracy by window - regional

* Is the accuracy defined by the center of effect within a span of notes (w_s), that is centered in the measure of referrence
* Key can be estimated with a window of > 1 measure bar (4 quarters for regular measure), but it is centered at a particular measure of referrence
* Guessed keys are compared with the "local key" annotation only, which refers to the relative numeral to the fundamental key (main key of the piece)

In [ ]:
num_p = 1
df_piece = copy(moz_dfs[num_p]);
df_ann = copy(moz_piece_ann[num_p]);


In [ ]:
f_key = zeros(Float64, length(moz_dfs))
s_key = zeros(Float64, length(moz_dfs))
t_key = zeros(Float64, length(moz_dfs))
e_counts = zeros(Int, length(moz_dfs))
for mp in good_ix
    try
        f_key[mp], s_key[mp], t_key[mp], e_counts[mp] = get_chord_accuracy_by_measure(moz_dfs[mp], moz_piece_ann[mp], include_keys=true)
    catch e
        println("Error processing piece $mp: $e")
    end
end


In [ ]:
fig = plot_violin_scatter(f_key[good_ix], s_key[good_ix], t_key[good_ix], title="Chord Accuracy for Mozart Pieces", labels=["1st Guess", "2nd Guess", "3rd Guess"])

In [ ]:
PlotlyJS.savefig(fig, "mozart_chord_accuracy.png")

In [ ]:
bee_ann_path = "/DCML_Corpora/ABC-2f12c12ab57ace9fb8d94b5f49ab1d7cca5db3dd/harmonies" #annotated data
bee_ann_list = readdir(bee_ann_path)
bee_ann_list = filter(x -> occursin(r"\.tsv$", x), bee_ann_list)
bee_ann_names = map(x -> split(x, ".")[1], bee_ann_list)
bee_piece_ann = map(x -> CSV.read(joinpath(bee_ann_path, bee_ann_list[x]), DataFrame, delim='\t', header=true), 1:length(bee_ann_list));

In [ ]:
bee_folder = "/DCML_Corpora/ABC-2f12c12ab57ace9fb8d94b5f49ab1d7cca5db3dd/notes"
bee_list = readdir(bee_folder)
bee_list = filter(x -> endswith(x, ".tsv"), bee_list);

In [ ]:
bee_names = map(x -> split(x, ".")[1], bee_list);

In [ ]:
bee_dfs = DataFrame[]
good_ix = Int64[]
for i in 1:length(bee_list)
    try
        tsv_piece = CSV.read(joinpath(bee_folder, bee_list[i]), DataFrame, delim='\t', header=true)
        df_piece = process_tsv_score(tsv_piece, repeats=true)
        push!(bee_dfs, df_piece)
        push!(good_ix, i)
    catch e
        println("Error processing $i: $e")
    end
end

In [ ]:
f_key = zeros(Float64, length(bee_dfs))
s_key = zeros(Float64, length(bee_dfs))
t_key = zeros(Float64, length(bee_dfs))
e_counts = zeros(Int, length(bee_dfs))
for mp in good_ix
    try
        piece_measures = groupby(bee_dfs[mp], :Measure)
        grouped_ann = groupby(bee_piece_ann[mp], :mc)
        f_key[mp], s_key[mp], t_key[mp], e_counts[mp] = get_chord_accuracy_by_measure(bee_dfs[mp], bee_piece_ann[mp], include_keys=true)
    catch e
        println("Error processing piece $mp: $e")
    end
end


In [ ]:
fig = plot_violin_scatter(f_key[good_ix], s_key[good_ix], t_key[good_ix], title="Chord Accuracy for Beethoven Pieces", labels=["1st Guess", "2nd Guess", "3rd Guess"])

In [ ]:
PlotlyJS.savefig(fig, "beethoven_chord_accuracy.png")

In [ ]:
bee_wk_acc = []
for num_p in 1:length(bee_dfs)
    push!(bee_wk_acc, get_key_accuracy_by_window(bee_dfs[num_p], bee_piece_ann[num_p]; w_s=30)[3])
end
fig = plot_violin_scatter(bee_wk_acc, title="Beethoven accuracy by window", ylabel="Accuracy", xlabel="Beethoven pieces")

In [ ]:
PlotlyJS.savefig(fig, "beethoven_accuracy_by_window.png")

In [ ]:
good_ix =findall(x-> x==0, e_counts);

In [ ]:
fig = plot_violin_scatter(f_key[good_ix], s_key[good_ix], t_key[good_ix], title="Key Accuracy for Beethoven Pieces", labels=["1st Guess", "2nd Guess", "3rd Guess"])

In [ ]:
PlotlyJS.savefig(fig, "beethoven_key_accuracy.html")

In [ ]:
ix = 1
df_calc = copy(moz_dfs[perr_ix[ix]])
df_ann = copy(moz_piece_ann[perr_ix[ix]])
piece_measures = groupby(df_calc, :Measure)
grouped_ann = groupby(df_ann, :mc);
pw = srt.get_oswindows(df_calc, 20, 19)
p_keys = Vector{String}(undef, length(pw))
for i in 1:length(pw)
    p_keys[i] = msr2.get_distance_to_keys(msr2.get_center_effect(Matrix(pw[i])))[1, 1]
end
piece_key = msr2.get_rank_freq(p_keys)[1, 1]
key_ann = Dict{Int,Vector{String}}()
for i in 1:length(grouped_ann)
    keys = vcat(convert(Vector{String}, collect(skipmissing(grouped_ann[i][!, :numeral]))), convert(Vector{String}, grouped_ann[i][!, :localkey]))
    num_me = grouped_ann[i][1, :mn]
    key_ann[num_me] = keys
end
key_guess = Dict{Int,Vector{String}}()
for i in 1:length(piece_measures)
    nm = piece_measures[i][1, :Measure]
    k_m = msr2.get_distance_to_keys(msr2.get_center_effect(Matrix(piece_measures[i])))[1:3, 1]
    key_guess[nm] = msr2.funhar_seq(k_m, piece_key)
end

In [ ]:
for key in keys(key_ann)
    try
        key_guess[key]
    catch
        println("Measure $key: Not found in key_guess.")
    end
end